In [1]:
import pandas as pd
import json
import os
import re
import ast
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [17]:
full_comments = pd.read_csv("../data/full_comments.csv")
examples = full_comments.loc[full_comments['true_label'].notna(), ["comment_text", "true_label"]].sample(10)
negative_examples = pd.read_excel("negative_examples.xlsx", names=['comment_text', 'original_label', 'corrected_label'])

examples = examples.rename(columns={"comment_text": "comment", "true_label": "label"})

examples_json = json.dumps(examples.to_dict(orient="records"), indent=2)
negative_examples_json = json.dumps(negative_examples.to_dict(orient="records"), indent=2)

In [18]:
subset = full_comments.sample(30000, random_state=10)
indices = [a for a in full_comments.index if a not in subset.index]
subset = full_comments.loc[indices, :]
subset['label'] = subset['true_label']

In [19]:
half = int(subset.shape[0] / 2)
subset1 = subset[:half]
subset2 = subset[half:]

In [20]:
COMMENT_COLUMN = "comment_text"
OUTPUT_CSV = "gemini_comments_labeled_p2.csv"

load_dotenv()
api_key = os.environ.get('GEMINI_API_KEY')

In [21]:
# ============================================================
# CONFIGURATION
# ============================================================

MODEL = "models/gemini-3-flash-preview"
BATCH_SIZE = 50
MAX_REQUESTS_PER_JOB = 100
JOB_NAMES_FILE = "gemini_job_names.json"

# ============================================================
# SYSTEM PROMPT
# ============================================================
SYSTEM_PROMPT = f"""You are a comment classifier. You will be given a batch of comments, each with an ID number. 
Classify each comment into exactly ONE of these five categories:

**Argumentative**
- Makes specific claims, predictions, or assertions supported by reasoning
- Uses evidence, anecdotes, or scenarios to build a case
- The key distinction from Opinion: there's an attempt to *persuade* or *explain why*, not just state a position

**Informational**
- Shares facts, data, links, or context relevant to the discussion
- Low emotional affect — the comment is trying to *inform*, not convince or react
- Includes answering another commenter's question with factual content
- The key distinction from Argumentative: presenting information without advocating for a position

**Opinion**
- States a value judgment, stance, or take without substantial reasoning
- "This is good/bad/wrong/overrated" — the comment *asserts* but doesn't *argue*
- The key distinction from Argumentative: no real attempt to persuade or support the claim
- The key distinction from Expressive: the comment is making a point, not just reacting

**Expressive**
- Emotional reactions, sarcasm, jokes, venting, exclamations
- The comment is primarily *expressing feeling* rather than making a point
- Includes performative agreement/disagreement ("THIS," "lol exactly," "what a joke")
- The key distinction from Opinion: no identifiable stance being taken, just affect

**Neutral**
- Clarifying or rhetorical questions, meta-commentary, off-topic remarks
- Comments that don't clearly fit the other four categories
- Includes simple factual questions directed at other commenters

**Correctly labeled examples** — these demonstrate the correct label for each comment:
{examples_json}

**Incorrectly labeled examples** — these were originally mislabeled. The "original_label" is the wrong label that was assigned, and the "corrected_label" is what the label should have been. Use these to understand common mistakes to avoid:
{negative_examples_json}

Respond with ONLY a valid JSON array where each element has "id", "label" keys and a confidence indicator where 
0 is not confident in the chosen label and 1 is confident in the chosen label.
Example: [{{"id": 0, "label": "Argumentative", "confidence": 1}}, {{"id": 1, "label": "Expressive", "confidence": 0}}]

Do not include any text outside the JSON array. No explanations, no markdown."""

VALID_LABELS = {"Argumentative", "Informational", "Opinion", "Expressive", "Neutral"}

def format_batch(comments):
    lines = []
    for idx, comment in comments:
        truncated = comment[:1500] if len(comment) > 1500 else comment
        lines.append(f"[{idx}] {truncated}")
    return "\n\n".join(lines)


def parse_response(response_text, expected_ids):
    if not response_text:
        return {}
    
    text = response_text.strip()

    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0]

    try:
        results = json.loads(text)
    except json.JSONDecodeError:
        try:
            results = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            match = re.search(r'\[.*\]', text, re.DOTALL)
            if match:
                try:
                    results = json.loads(match.group())
                except json.JSONDecodeError:
                    return {}
            else:
                return {}

    if results and isinstance(results[0], list):
        results = results[0]

    labels = {}

    for item in results:
        idx = item.get("id")
        label = item.get("label")

        if not isinstance(label, str):
            continue

        label = label.strip()
        conf = item.get("confidence", None)

        if idx not in expected_ids:
            continue

        if label not in VALID_LABELS:
            matched = [v for v in VALID_LABELS if v.lower() == label.lower()]
            if matched:
                label = matched[0]
            else:
                continue
        labels[idx] = {"label": label, "confidence": conf}
    return labels

def save_results(df):
    df.to_csv(OUTPUT_CSV, index=False)
    return df

client = genai.Client(api_key=api_key)

In [22]:
# ============================================================
# STEP 1: Build inline requests and submit batch jobs
#         (100 requests per job due to tier limit)
# ============================================================

df = subset2.copy()

# Build batches of comments
unlabeled_mask = df["label"].isna()
unlabeled_indices = df[unlabeled_mask].index.tolist()
print(f"{len(unlabeled_indices)} comments to label")

batches = []
for i in range(0, len(unlabeled_indices), BATCH_SIZE):
    batch_indices = unlabeled_indices[i:i + BATCH_SIZE]
    batch = [(idx, str(df.loc[idx, COMMENT_COLUMN])) for idx in batch_indices]
    batches.append(batch)

print(f"{len(batches)} requests to submit")

# Build inline requests
all_requests = []
batch_mapping = {}
for i, batch in enumerate(batches):
    expected_ids = [idx for idx, _ in batch]
    batch_mapping[str(i)] = expected_ids
    all_requests.append({
        'contents': [{
            'parts': [{'text': format_batch(batch)}],
            'role': 'user'
        }],
        'config': {
            'system_instruction': {'parts': [{'text': SYSTEM_PROMPT}]},
            'thinking_config': {'thinking_level': 'minimal'}
        }
    })

# Submit in groups of MAX_REQUESTS_PER_JOB
job_names = []
for chunk_start in range(0, len(all_requests), MAX_REQUESTS_PER_JOB):
    chunk = all_requests[chunk_start:chunk_start + MAX_REQUESTS_PER_JOB]
    chunk_num = chunk_start // MAX_REQUESTS_PER_JOB

    batch_job = client.batches.create(
        model=MODEL,
        src=chunk,
        config={'display_name': f'labeling-chunk-{chunk_num}'}
    )
    job_names.append(batch_job.name)
    print(f"  Chunk {chunk_num}: submitted {len(chunk)} requests -> {batch_job.name}")

# Save job names and mapping for retrieval later
with open(JOB_NAMES_FILE, "w") as f:
    json.dump({"job_names": job_names, "batch_mapping": batch_mapping}, f)

print(f"\n{len(job_names)} batch jobs submitted")
print(f"Saved to {JOB_NAMES_FILE}")
print("You can close your computer now.")

23819 comments to label
477 requests to submit
  Chunk 0: submitted 100 requests -> batches/oiq4048jlr4i2v1rkmj3khk1fjczpxusiprk
  Chunk 1: submitted 100 requests -> batches/ql7evdm2txi1x05m422xrezcbz2dxpywh4mw
  Chunk 2: submitted 100 requests -> batches/52bzwozzt3kgy58gwzaf6c4g01x0oxnen1tj
  Chunk 3: submitted 100 requests -> batches/hzov96g8n0ntsq62uebf8pzybw5ilkbcvbjj
  Chunk 4: submitted 77 requests -> batches/wxdvo1rn3vh3osj15756s1su92g8w14sb47i

5 batch jobs submitted
Saved to gemini_job_names.json
You can close your computer now.


In [34]:
# ============================================================
# STEP 2: Check batch status (run when you come back)
# ============================================================

with open(JOB_NAMES_FILE, "r") as f:
    saved = json.load(f)
    job_names = saved["job_names"]

for name in job_names:
    job = client.batches.get(name=name)
    print(f"{job.name}: {job.state.name}")

batches/oiq4048jlr4i2v1rkmj3khk1fjczpxusiprk: JOB_STATE_SUCCEEDED
batches/ql7evdm2txi1x05m422xrezcbz2dxpywh4mw: JOB_STATE_SUCCEEDED
batches/52bzwozzt3kgy58gwzaf6c4g01x0oxnen1tj: JOB_STATE_SUCCEEDED
batches/hzov96g8n0ntsq62uebf8pzybw5ilkbcvbjj: JOB_STATE_SUCCEEDED
batches/wxdvo1rn3vh3osj15756s1su92g8w14sb47i: JOB_STATE_SUCCEEDED


In [35]:
# ============================================================
# STEP 2.5: Check for errors in completed jobs
# ============================================================

with open(JOB_NAMES_FILE, "r") as f:
    saved = json.load(f)
    job_names = saved["job_names"]

for name in job_names:
    job = client.batches.get(name=name)

    if job.state.name == 'JOB_STATE_FAILED':
        print(f"{name}: FAILED")
        if hasattr(job, 'error') and job.error:
            print(f"  Error: {job.error}")
        continue

    if job.state.name != 'JOB_STATE_SUCCEEDED':
        print(f"{name}: {job.state.name} (still running)")
        continue

    # Check individual request errors within succeeded jobs
    error_count = 0
    for i, inline_response in enumerate(job.dest.inlined_responses):
        if inline_response.error:
            error_count += 1
            if error_count <= 5:
                print(f"  {name} request {i}: {inline_response.error}")

    if error_count > 5:
        print(f"  ... and {error_count - 5} more errors")
    elif error_count == 0:
        print(f"{name}: all requests succeeded")

batches/oiq4048jlr4i2v1rkmj3khk1fjczpxusiprk: all requests succeeded
batches/ql7evdm2txi1x05m422xrezcbz2dxpywh4mw: all requests succeeded
batches/52bzwozzt3kgy58gwzaf6c4g01x0oxnen1tj: all requests succeeded
batches/hzov96g8n0ntsq62uebf8pzybw5ilkbcvbjj: all requests succeeded
batches/wxdvo1rn3vh3osj15756s1su92g8w14sb47i: all requests succeeded


In [36]:
# ============================================================
# STEP 3: Retrieve results and save
#         (run once all jobs show JOB_STATE_SUCCEEDED)
# ============================================================

with open(JOB_NAMES_FILE, "r") as f:
    saved = json.load(f)
    job_names = saved["job_names"]
    batch_mapping = saved["batch_mapping"]

df = subset2.copy()
total_labeled = 0
failed = 0
request_idx = 0
debug_responses = []

for name in job_names:
    job = client.batches.get(name=name)

    if job.state.name != 'JOB_STATE_SUCCEEDED':
        print(f"  Skipping {name} — state: {job.state.name}")
        chunk_size = min(MAX_REQUESTS_PER_JOB, len(batch_mapping) - request_idx)
        request_idx += chunk_size
        continue

    for inline_response in job.dest.inlined_responses:
        expected_ids = batch_mapping[str(request_idx)]

        if inline_response.response:
            response_text = inline_response.response.text

            # Handle None or empty text (e.g. thinking consued all tokens)
            if not response_text:
                failed += 1
                debug_responses.append({
                    "request_idx": request_idx,
                    "job_name": name,
                    "issue": "response.text is None/empty",
                    "response_obj": str(inline_response.response)
                })
                print(f"  Request {request_idx}: response.text is None - saved for debug")
                request_idx += 1
                continue

            try:
                labels = parse_response(response_text, expected_ids)
            except (AttributeError, TypeError) as e:
                failed += 1
                debug_responses.append({
                    "requests_idx": request_idx,
                    "job_name": name,
                    "issue": str(e),
                    "response_text": response_text[:500] if response_text else None,
                    "response_obj": str(inline_response.response)
                })
                print(f"  Request {request_idx}: parse error ({e}) - saved for debug")
                request_idx += 1
                continue

            for idx, value in labels.items():
                df.loc[idx, "label"] = value["label"]
                df.loc[idx, "confidence"] = value["confidence"]

            total_labeled += len(labels)
        else:
            failed += 1
            print(f"  Request {request_idx} failed: {inline_response.error}")

        request_idx += 1

combined = save_results(df)

print(f"\nDONE - {total_labeled} comments labeled, {failed} requests failed")
print(f"Saved to: {OUTPUT_CSV} ({len(combined)} total rows)")
print(f"\nLabel distribution:")
print(df["label"].value_counts().to_string())

if debug_responses:
    print(f"\n{len(debug_responses)} problematic responses saved to 'debug_responses' variable")
    with open("gemini_debug_responses.json", "w") as f:
        json.dump(debug_responses, f, indent=2)
    print("Also saved to gemini_debug_responses.json")


DONE - 23666 comments labeled, 0 requests failed
Saved to: gemini_comments_labeled_p2.csv (23857 total rows)

Label distribution:
label
Expressive       9340
Opinion          9067
Argumentative    3160
Neutral          1390
Informational     747


In [39]:
# Load files
gemini_p1 = pd.read_csv("gemini_comments_labeled_p1.csv")
gemini_p2 = pd.read_csv("gemini_comments_labeled_p2.csv")

# Stack rows
gemini_df = pd.concat(
    [gemini_p1, gemini_p2],
    ignore_index=True
)

# Optional: remove exact duplicates (recommended)
combined_gemini_df = gemini_df.drop_duplicates()

# Save result
combined_gemini_df.to_csv("gemini_labeled_full.csv", index=False)

print("Combination complete.")
print("New shape:", combined_gemini_df.shape)
print("\nColumns:")
print(combined_gemini_df.columns.tolist())

Combination complete.
New shape: (46914, 5)

Columns:
['comment_text', 'source', 'true_label', 'label', 'confidence']


In [40]:
# Load datasets
rest_df = pd.read_csv("rest_of_ai_labeled_comments.csv")

# Keep only needed Gemini columns and rename
gemini_subset = combined_gemini_df[["comment_text", "label", "confidence"]].copy()

gemini_subset = gemini_subset.rename(columns={
    "label": "gemini_label",
    "confidence": "gemini_confidence"
})

# Merge on comment_text
merged_df = rest_df.merge(
    gemini_subset,
    on="comment_text",
    how="left"   # keeps all rows from rest_df
)

# Save result
merged_df.to_csv("all_ai_labeled_comments.csv", index=False)

print("Merge complete.")
print("New shape:", merged_df.shape)
print("\nColumns:")
print(merged_df.columns.tolist())

Merge complete.
New shape: (49108, 8)

Columns:
['comment_text', 'source', 'openai_label', 'openai_confidence', 'claude_labels', 'claude_confidence', 'gemini_label', 'gemini_confidence']


## Preprocess

In [10]:
from preprocessing_script import preprocess_text

In [49]:
df = pd.read_csv("combined_with_gemini_30k.csv")

In [50]:
import re
import emoji # type: ignore
import nltk
import html
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure necessary NLTK resources are available
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def preprocess_text(text, full=False):
    # 1. Basic Cleaning (Good for BERT)
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)  # Remove HTML
    text = re.sub(r'http\S+', '[URL]', text)  # Replace URLs
    text = html.unescape(text) # Convert escaped characters back to original (&quot; -> ")
    
    # 2. Handle Emojis (Convert to text descriptions)
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 3. 'Full' Processing (For Logistic Regression / Traditional ML)
    if full:
        # Define allowed punctuation: keep ? and !
        # This regex removes punctuation EXCEPT ? and !
        text = re.sub(r'[^\w\s\?\!]', '', text)
        
        # Tokenize for processing
        words = text.split()
        
        # Initialize Stopwords and Lemmatizer
        stop_words = set(stopwords.words('english'))
        lemmatizer = WordNetLemmatizer()
        
        # Filter stopwords and Lemmatize
        words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
        
        text = " ".join(words)
    
    # 4. Final Cleanup of extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [51]:
df["comment_text"] = df["comment_text"].apply(
    lambda x: preprocess_text(x, full=False)
)

df.to_csv("combined_with_gemini_30k.csv", index=False)

## Majority Vote Labeling

In [52]:
from collections import Counter

df = pd.read_csv("combined_with_gemini_30k.csv")

def majority_vote(row):
    votes = [
        row["label"],
        row["openai_label"],
        row["gemini_label"]
    ]
    
    if len(votes) < 3:
        return "Review"
    
    counts = Counter(votes)
    most_common = counts.most_common()
    
    # If top vote count is 1 → all different (3-way tie)
    if most_common[0][1] == 1:
        return "Review"
    
    return most_common[0][0]

df["true_label"] = df.apply(majority_vote, axis=1)

df.to_csv("combined_with_differencing.csv", index=False)

print("Done.")
print(df["true_label"].value_counts())

Done.
true_label
Expressive       9410
Opinion          8656
Argumentative    5860
Neutral          2844
Informational    2007
Review           1223
Name: count, dtype: int64


In [36]:
# Identify true 3-way disagreements
three_way_ties = df[
    df[["label", "openai_label", "gemini_label"]]
    .nunique(axis=1) == 3
]

print("Total 3-way ties:", len(three_way_ties))

# Count combinations
combo_counts = (
    three_way_ties
    .groupby(["label", "openai_label", "gemini_label"])
    .size()
    .sort_values(ascending=False)
)

print("\nLargest 3-way tie combination:")
print(combo_counts.head(1))

print("\nTop 10 3-way tie combinations:")
print(combo_counts.head(10))

Total 3-way ties: 1180

Largest 3-way tie combination:
label    openai_label  gemini_label
Neutral  Opinion       Expressive      126
dtype: int64

Top 10 3-way tie combinations:
label          openai_label   gemini_label 
Neutral        Opinion        Expressive       126
Opinion        Argumentative  Expressive       108
Expressive     Argumentative  Opinion          101
Neutral        Argumentative  Opinion           96
               Informational  Opinion           91
               Expressive     Opinion           91
Opinion        Expressive     Argumentative     57
Informational  Argumentative  Opinion           39
Opinion        Argumentative  Neutral           31
               Informational  Argumentative     30
dtype: int64


In [37]:
# Full agreement = only 1 unique value across the 3 columns
full_agreement_mask = (
    df[["label", "openai_label", "gemini_label"]]
    .nunique(axis=1) == 1
)

full_agreement_count = full_agreement_mask.sum()
total = len(df)

percentage = (full_agreement_count / total) * 100

print(f"Full agreement count: {full_agreement_count}")
print(f"Total records: {total}")
print(f"Full agreement percentage: {percentage:.2f}%")

Full agreement count: 17687
Total records: 30000
Full agreement percentage: 58.96%


## Pre-training for Hugging Face

In [19]:
from datasets import Dataset, DatasetDict, Features, ClassLabel, Value

def train_prep(df: pd.DataFrame, label_column: str = "true_label", test1: float=0.3, test2: float=0.5):
    
    # Remove rows marked for review and ambiguous labels
    df = df[df[label_column] != "Review"].copy()
    df = df[df[label_column].notna()].copy()
    df = df[df["comment_text"].notna()].copy()

    # Correct types
    df[label_column] = df[label_column].astype(str)
    df["comment_text"] = df["comment_text"].astype(str)

    # Label mapping
    unique_labels = sorted(df[label_column].unique())
    label2id = {label: idx for idx, label in enumerate(unique_labels)}
    id2label = {idx: label for label, idx in label2id.items()}

    # Prepare dataset
    dataset = Dataset.from_dict({
        "text": df['comment_text'].tolist(),
        "label": [label2id[label] for label in df[label_column]]
    })

    class_features = Features({
        'text': Value('string'),
        'label': ClassLabel(names=unique_labels) # This creates the id2label mapping
    })

    # Cast your dataset to these features
    dataset = dataset.cast(class_features)

    # Split dataset 3-ways and combine back into  one
    split_dataset = dataset.train_test_split(test_size=test1, seed=10)
    test_valid = split_dataset['test'].train_test_split(test_size=test2, seed=10)

    final_dataset = DatasetDict({
        "train": split_dataset['train'],
        "test": test_valid['test'],
        "validation": test_valid['train']
    })

    return final_dataset, label2id, id2label

## Final dataset for fine-tuning

In [20]:
df = pd.read_csv("combined_with_differencing.csv")

dataset_dict, label2id, id2label = train_prep(df)

print(dataset_dict)
print(label2id)

Casting the dataset:   0%|          | 0/28699 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20089
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 4305
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 4305
    })
})
{'Argumentative': 0, 'Expressive': 1, 'Informational': 2, 'Neutral': 3, 'Opinion': 4}


In [21]:
dataset_dict.push_to_hub(
    "ADS509/final_project_data",
    data_dir="experiment-labels",
    private=False
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Repo card metadata block was not found. Setting CardData to empty.
/opt/miniconda3/envs/ads502/lib/python3.10/site-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


CommitInfo(commit_url='https://huggingface.co/datasets/ADS509/final_project_data/commit/7b5612b2aa5de643654e0fd18b4e1cf2ec447ebf', commit_message='Upload dataset', commit_description='', oid='7b5612b2aa5de643654e0fd18b4e1cf2ec447ebf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ADS509/final_project_data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ADS509/final_project_data'), pr_revision=None, pr_num=None)

In [24]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import os

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

MODEL_ID = 'bert-base-uncased'

# check for gpu
torch.cuda.is_available()

False

In [23]:
# Load the dataset you're using, don't forget to specify the data directory
dataset = load_dataset('ADS509/final_project_data', data_dir="experiment-labels")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Function to tokenize data with
def tokenize_function(batch):
    return tokenizer(
        batch['text'],
        truncation=True, 
       # padding='max_length',
       # max_length=128 # Can't be greater than model max length
    )
# Data collator handles padding dynamically, set padding and max_length if you want to control it explicitly and drop the collator

# Tokenize Data
train_data = dataset['train'].map(tokenize_function, batched=True, remove_columns=dataset['train'].column_names)
test_data = dataset['test'].map(tokenize_function, batched=True, remove_columns=dataset['test'].column_names)
valid_data = dataset['valid'].map(tokenize_function, batched=True, remove_columns=dataset['valid'].column_names)

# Convert lists to tensors
train_data.set_format("torch", columns=['input_ids', "attention_mask", "label"])
test_data.set_format("torch", columns=['input_ids', "attention_mask", "label"])
valid_data.set_format("torch", columns=['input_ids', "attention_mask", "label"])

label_list = dataset["train"].features["label"].names

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

     
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=5, # adjust this based on number of labels you're training on
    dtype='auto',
    label2id={}, # set these two args to attach the metadata to the model.config
    id2label={}
)

# Metric function for evaluation in Trainer
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

# Data collator to handle padding dynamically per batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='./bert-comment-classifier', # Saves it locally
    push_to_hub=True,
    hub_model_id="ADS509/final_project_models",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,  # or warmup_steps=some int
    
    # Evaluation & saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    
    # Logging
    logging_steps=100,
    report_to='tensorboard',
    
    # Other
    seed=42,
    fp16=torch.cuda.is_available(),  # Mixed precision if GPU available
)
     

# Set up Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train!
trainer.train()

# Evaluate
eval_results = trainer.evaluate()
print(eval_results)
     

# Save trained model to hugging face model repo
trainer.save_model(training_args.output_dir)
trainer.push_to_hub(commit_message = "")

Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/4305 [00:00<?, ? examples/s]

ValueError: You passed along `num_labels=5` with an incompatible id to label map: {}. Since those arguments are inconsistent with each other, you should remove one of them.